In [ ]:


import os, re, json, gc
from pathlib import Path
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer



In [ ]:
# ----------------------------
# 1) Configuration
# ----------------------------

# Where your input CSV is. You can set PATENTS_CSV env var instead.
DATA_PATH = os.environ.get("PATENTS_CSV", "patents_detailed.csv")

# Where outputs go (JSONL + CSV). You can set PATENTS_OUT env var instead.
OUT_DIR = Path(os.environ.get("PATENTS_OUT", "patent_tagging_outputs"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Hugging Face cache location (optional). You can set HF_CACHE env var instead.
HF_CACHE_DIR = os.environ.get("HF_CACHE", None)
if HF_CACHE_DIR:
    os.environ["HF_HOME"] = HF_CACHE_DIR
    os.environ["TRANSFORMERS_CACHE"] = HF_CACHE_DIR

# Set True to delete existing outputs for a clean rerun.
RESET_EXISTING_OUTPUTS = False

# Rows limit for quick tests
MAX_ROWS = None  # e.g. 50

# Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("DATA_PATH:", DATA_PATH)
print("OUT_DIR   :", str(OUT_DIR))
print("DEVICE    :", DEVICE)
if DEVICE == "cuda":
    print("GPU       :", torch.cuda.get_device_name(0))

In [ ]:


# ----------------------------
# 2) Load dataset
# ----------------------------
DATA_PATH = Path(DATA_PATH)
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Could not find input CSV: {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH)
if "abstract" not in df.columns:
    raise KeyError(f"'abstract' column not found. Columns: {list(df.columns)}")

print("Loaded:", str(DATA_PATH.resolve()))
print("Shape :", df.shape)

# ----------------------------
# 3) Prompt template (kept for reference only)
# ----------------------------
DRUG_DEV_PROMPT_TEMPLATE = """
You are an expert in biomedical innovation, pharmacology, and patent analysis.

Task:
Determine whether the following patent is related to MEDICINE OR DRUG DEVELOPMENT.

Definition:
A patent is considered medicine/drug-development-related if its primary aim
concerns the discovery, development, optimization, validation, or therapeutic
use of medicines. This includes (but is not limited to):

- Identification or validation of therapeutic targets
- Small-molecule drugs, biologics, gene or cell therapies
- Drug repurposing
- Biomarkers explicitly used for treatment selection or stratification
- Therapeutic compositions, formulations, dosing, or delivery systems
- Methods of treatment or prevention of disease using a therapeutic agent

Exclude:
- Purely diagnostic inventions without therapeutic application
- Basic biological research without translational or therapeutic intent
- Medical devices not directly linked to drug delivery
- General research tools or assays without a drug development focus

Output strictly in valid JSON using the following schema:
{{"label": "yes" or "no",
  "development_stage": "target discovery" | "lead discovery" | "preclinical" | "clinical/therapeutic use" | "unclear",
  "confidence": "high" | "medium" | "low",
  "rationale": "1–2 sentences explaining the decision"}}

Patent abstract:
<<<
{abstract}
>>>
"""

# ----------------------------
# 4) Helpers (schema unchanged)
# ----------------------------
TAG_RE = re.compile(r"<[^>]+>")

def clean_abstract(x: str) -> str:
    if pd.isna(x):
        return ""
    x = str(x)
    x = TAG_RE.sub(" ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

ALLOWED_STAGE = {
    "target discovery",
    "lead discovery",
    "preclinical",
    "clinical/therapeutic use",
    "unclear",
}
ALLOWED_CONF = {"high", "medium", "low"}
ALLOWED_LABEL = {"yes", "no"}

def normalise_and_validate(obj):
    fallback = {
        "label": "no",
        "development_stage": "unclear",
        "confidence": "low",
        "rationale": "Could not parse/validate model output as required JSON."
    }
    if not isinstance(obj, dict):
        return fallback

    label = str(obj.get("label", "")).strip().lower()
    stage = str(obj.get("development_stage", "")).strip().lower()
    conf  = str(obj.get("confidence", "")).strip().lower()
    rat   = str(obj.get("rationale", "")).strip()

    if label in ("y", "true"): label = "yes"
    if label in ("n", "false"): label = "no"
    if stage == "clinical": stage = "clinical/therapeutic use"
    if stage == "therapeutic use": stage = "clinical/therapeutic use"

    if label not in ALLOWED_LABEL: label = "no"
    if stage not in ALLOWED_STAGE: stage = "unclear"
    if conf not in ALLOWED_CONF: conf = "low"
    if not rat: rat = "No rationale provided."

    return {
        "label": label,
        "development_stage": stage,
        "confidence": conf,
        "rationale": rat
    }

# ----------------------------
# 5) Embedding backends
# ----------------------------

def _mean_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

@torch.inference_mode()
def embed_scibert(tok, model, texts, max_length=512, batch_size=64, dtype=torch.float16):
    embs = []
    for i in range(0, len(texts), batch_size):
        chunk = texts[i:i+batch_size]
        inputs = tok(
            chunk,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length,
        )
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        use_amp = (DEVICE == "cuda") and (dtype in (torch.float16, torch.bfloat16))
        with torch.autocast(device_type="cuda", dtype=dtype, enabled=use_amp):
            out = model(**inputs)
            pooled = _mean_pool(out.last_hidden_state, inputs["attention_mask"])

        pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
        embs.append(pooled.detach().cpu())
    return torch.cat(embs, dim=0)

def embed_sbert(st: SentenceTransformer, texts, batch_size=128):
    return st.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=False,
        convert_to_tensor=True,
        normalize_embeddings=True,
    ).cpu()

def load_embedder(model_id: str, kind: str, dtype: torch.dtype, max_input_tokens: int):
    """
    kind: "scibert" or "sbert"
    Returns (embed_fn, state)
    """
    if kind == "scibert":
        tok = AutoTokenizer.from_pretrained(model_id, use_fast=True)
        model = AutoModel.from_pretrained(model_id).to(DEVICE)
        model.eval()

        def _embed(texts, max_length=max_input_tokens, batch_size=64):
            return embed_scibert(tok, model, texts, max_length=max_length, batch_size=batch_size, dtype=dtype)

        return _embed, {"tokenizer": tok, "model": model}

    elif kind == "sbert":
        st = SentenceTransformer(model_id, device=DEVICE)
        st.max_seq_length = int(max_input_tokens)

        def _embed(texts, max_length=max_input_tokens, batch_size=128):
            return embed_sbert(st, texts, batch_size=batch_size)

        return _embed, {"st": st}

    else:
        raise ValueError(f"Unknown kind: {kind}")

# ----------------------------
# 6) Prototypes + scoring (schema preserved)
# ----------------------------
LABEL_PROTOTYPES = {
    "yes": "Therapeutic invention: drug candidates, biologics, antibodies, inhibitors, formulations, dosing, delivery, treatment, pharmacology, clinical use.",
    "no":  "Non-therapeutic invention: diagnostics only, imaging, sensors, devices, materials, manufacturing, software tools, assays without treatment focus."
}

STAGE_PROTOTYPES = {
    "target discovery": "Target discovery: identifying or validating therapeutic targets, pathways, disease mechanisms, biomarkers for treatment selection.",
    "lead discovery": "Lead discovery: screening, hit-to-lead, optimisation of compounds/biologics, structure-activity relationships, candidate selection.",
    "preclinical": "Preclinical: in vitro or in vivo efficacy, pharmacokinetics, toxicity, animal models, formulation optimisation before human trials.",
    "clinical/therapeutic use": "Clinical/therapeutic use: methods of treatment, dosing regimens, patient administration, therapeutic compositions used in humans.",
    "unclear": "Unclear development stage."
}

STOPWORDS = set("""
a an and are as at be been being by can could did do does doing for from had has have having how if in into is it its
may might more most not of on or our should such than that the their then there these they this those to was were what
when where which who will with within without you your we
""".split())

def top_terms(text: str, k: int = 4):
    words = re.findall(r"[A-Za-z][A-Za-z\-]{2,}", (text or "").lower())
    words = [w for w in words if w not in STOPWORDS]
    if not words:
        return []
    freq = {}
    for w in words:
        freq[w] = freq.get(w, 0) + 1
    ranked = sorted(freq.items(), key=lambda x: (x[1], len(x[0])), reverse=True)
    out = []
    for w, _ in ranked:
        if w not in out:
            out.append(w)
        if len(out) >= k:
            break
    return out

def score_to_confidence(margin: float):
    if margin >= 0.06:
        return "high"
    if margin >= 0.03:
        return "medium"
    return "low"

TASK_QUERY = (
    "Classify whether the patent is related to medicine or drug development (yes/no). "
    "If yes, infer stage: target discovery, lead discovery, preclinical, or clinical/therapeutic use. "
    "If no, stage is unclear."
)

def build_embedding_text(abstract_text: str) -> str:
    # abstract first, then short query (prevents truncation from dropping the abstract)
    return f"Abstract:\n{abstract_text}\n\n{TASK_QUERY}"

# ----------------------------
# 7) Main runner (resume-friendly)
# ----------------------------
def run_model_tagging_fast(
    model_id: str,
    model_tag: str,
    model_kind: str,         # "scibert" or "sbert"
    df_in: pd.DataFrame,
    *,
    max_rows: int | None = None,
    batch_size: int = 64,
    max_input_tokens: int = 512,
    dtype: torch.dtype = torch.float16,
):
    print("\n" + "="*90)
    print(f"Model: {model_id}")
    print(f"Tag  : {model_tag} | kind={model_kind}")
    print(f"BS   : {batch_size} | max_input_tokens={max_input_tokens} | dtype={dtype} | device={DEVICE}")
    print("="*90)

    jsonl_path = OUT_DIR / f"preds_{model_tag}.jsonl"
    csv_path   = OUT_DIR / f"patents_tagged_{model_tag}.csv"

    if RESET_EXISTING_OUTPUTS:
        if jsonl_path.exists():
            jsonl_path.unlink()
            print(f"Reset: deleted {jsonl_path}")
        if csv_path.exists():
            csv_path.unlink()

    # Resume support
    done_idx = set()
    if jsonl_path.exists():
        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    done_idx.add(int(rec["row_index"]))
                except Exception:
                    continue
        print(f"Resume: {len(done_idx)} rows already in {jsonl_path}")

    embed_fn, state = load_embedder(model_id, model_kind, dtype, max_input_tokens)

    label_keys = list(LABEL_PROTOTYPES.keys())
    label_texts = [LABEL_PROTOTYPES[k] for k in label_keys]
    stage_keys = list(STAGE_PROTOTYPES.keys())
    stage_texts = [STAGE_PROTOTYPES[k] for k in stage_keys]

    label_emb = embed_fn(label_texts, max_length=max_input_tokens, batch_size=min(batch_size, len(label_texts)))
    stage_emb = embed_fn(stage_texts, max_length=max_input_tokens, batch_size=min(batch_size, len(stage_texts)))

    n = len(df_in) if max_rows is None else min(len(df_in), max_rows)

    # Guard: if fully done, exit early (prevents "instant run" confusion)
    if len(done_idx) >= n:
        print(f"Nothing to do: {len(done_idx)}/{n} rows already present in {jsonl_path}.")
        print("Set RESET_EXISTING_OUTPUTS=True to recompute from scratch.")
        # Still write CSV from existing JSONL (as before)
        pass

    # Load existing parsed results for CSV output
    pred_rows = [None] * n
    if jsonl_path.exists():
        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    i = int(rec["row_index"])
                    if i < n:
                        pred_rows[i] = rec["parsed"]
                except Exception:
                    continue

    jf = open(jsonl_path, "a", encoding="utf-8")

    try:
        pbar = tqdm(total=n, desc=f"Tagging ({model_tag})")

        i = 0
        while i < n:
            batch_indices = []
            while i < n and len(batch_indices) < batch_size:
                if i in done_idx:
                    pbar.update(1)
                else:
                    batch_indices.append(i)
                i += 1

            if not batch_indices:
                continue

            texts = [build_embedding_text(clean_abstract(df_in.loc[idx, "abstract"])) for idx in batch_indices]
            emb = embed_fn(texts, max_length=max_input_tokens, batch_size=min(batch_size, len(texts)))  # [B, D]

            sims_label = emb @ label_emb.T  # [B, 2]
            sims_stage = emb @ stage_emb.T  # [B, 5]

            for bi, idx in enumerate(batch_indices):
                sl = sims_label[bi].tolist()
                order = sorted(range(len(sl)), key=lambda j: sl[j], reverse=True)
                best_j, second_j = order[0], order[1]
                best_label = label_keys[best_j]
                margin = float(sl[best_j] - sl[second_j])
                conf = score_to_confidence(margin)

                if best_label == "yes":
                    ss = sims_stage[bi].tolist()
                    sj = max(range(len(ss)), key=lambda j: ss[j])
                    stage = stage_keys[sj]
                    if stage == "unclear" and conf != "high":
                        stage = "unclear"
                else:
                    stage = "unclear"

                abst = clean_abstract(df_in.loc[idx, "abstract"])
                terms = top_terms(abst, k=4)

                if best_label == "yes":
                    rat = (
                        f"Abstract discusses {', '.join(terms[:3])}, suggesting a therapeutic or drug-development focus."
                        if terms else
                        "Abstract suggests a therapeutic or drug-development focus based on semantic similarity."
                    )
                    if stage != "unclear":
                        rat = rat.rstrip(".") + f", consistent with a {stage} stage."
                else:
                    rat = (
                        f"Abstract mainly concerns {', '.join(terms[:3])}, with little indication of therapeutic agents or drug-development activity."
                        if terms else
                        "Abstract shows little indication of therapeutic agents or drug-development activity."
                    )

                parsed = normalise_and_validate({
                    "label": best_label,
                    "development_stage": stage,
                    "confidence": conf,
                    "rationale": rat
                })
                pred_rows[idx] = parsed

                jf.write(json.dumps({
                    "row_index": idx,
                    "model_id": model_id,
                    "model_tag": model_tag,
                    "parsed": parsed,
                    "raw_text": json.dumps({
                        "label_scores": dict(zip(label_keys, [float(x) for x in sl])),
                        "label_margin": margin,
                        "top_terms": terms[:6],
                    }, ensure_ascii=False)[:4000],
                }, ensure_ascii=False) + "\n")

            jf.flush()
            pbar.update(len(batch_indices))

        pbar.close()

    finally:
        jf.close()

        out_df = df_in.iloc[:n].copy()
        out_df["drugdev_label"] = [r["label"] if r else None for r in pred_rows]
        out_df["drugdev_development_stage"] = [r["development_stage"] if r else None for r in pred_rows]
        out_df["drugdev_confidence"] = [r["confidence"] if r else None for r in pred_rows]
        out_df["drugdev_rationale"] = [r["rationale"] if r else None for r in pred_rows]
        out_df["drugdev_model"] = model_id

        out_df.to_csv(csv_path, index=False)

        print(f"\nSaved JSONL: {jsonl_path}")
        print(f"Saved CSV : {csv_path}")
        print("\nLabel counts:")
        print(out_df["drugdev_label"].value_counts(dropna=False))

        # Cleanup
        try:
            if model_kind == "scibert":
                del state["model"], state["tokenizer"]
            else:
                del state["st"]
        except Exception:
            pass
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# ----------------------------
# 8) Run models
# ----------------------------
MODELS = [
    ("allenai/scibert_scivocab_uncased",       "scibert", "scibert", 96),
    ("sentence-transformers/all-MiniLM-L6-v2", "sbert",   "sbert",   256),
]

# dtype choice
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

for model_id, tag, kind, bs in MODELS:
    try:
        run_model_tagging_fast(
            model_id=model_id,
            model_tag=tag,
            model_kind=kind,
            df_in=df,
            max_rows=MAX_ROWS,
            batch_size=bs,
            max_input_tokens=512,
            dtype=DTYPE,
        )
    except Exception as e:
        print(f"\n[ERROR] Model {tag} failed: {type(e).__name__}: {e}")
        print("Continuing to next model...\n")

print("\nAll done. Outputs are in:", str(OUT_DIR.resolve()))
